# Sampling Method Comparison

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc
from scipy.spatial.distance import cdist

## Constants

In [30]:
N_SAMPLES  = 64      # Number of sample points
N_DIMS     = 4        # Number of design variables (W1, W2, R, t)
N_REPEATS  = 50       # Number of realisations for random sampling and LHS
SEED_BASE  = 42       # Base random seed for reproducibility

## Evaluation Metrics

In [ ]:
# AMD = Average Euclidean distance to nearest neighbour (higher = better space-filling)
def average_minimum_distance(samples):
    dist_matrix = cdist(samples, samples)
    np.fill_diagonal(dist_matrix, np.inf)  # exclude self-distances
    return np.mean(np.min(dist_matrix, axis=1))

# Discrepancy = L2* Star Discrepancy (lower = better uniformity)
def l2_star_discrepancy(samples):
    return qmc.discrepancy(samples, method='L2-star')

## Sampling Methods

In [ ]:
# Random Sampling: uniform random within the unit hypercube
def random_sampling(n_samples, n_dims, seed):
    rng = np.random.default_rng(seed)
    return rng.random((n_samples, n_dims))

# Sobol Sequence: low-discrepancy, deterministic — no seed needed
def sobol_sampling(n_samples, n_dims):
    sampler = qmc.Sobol(d=n_dims)
    return sampler.random(n_samples)

# Latin Hypercube Sampling: stratified — one sample per row/column band per dimension
def lhs_sampling(n_samples, n_dims, seed):
    sampler = qmc.LatinHypercube(d=n_dims, seed=seed)
    return sampler.random(n_samples)

## Generate samples and evaluate

### Random Sampling

In [ ]:
random_amd  = []  # Average Minimum Distance across realisations
random_disc = []  # L2* Discrepancy across realisations

for i in range(N_REPEATS):
    samples = random_sampling(N_SAMPLES, N_DIMS, seed=SEED_BASE + i)  # different seed per realisation
    random_amd.append(average_minimum_distance(samples))
    random_disc.append(l2_star_discrepancy(samples))

random_amd_mean, random_amd_std   = np.mean(random_amd),  np.std(random_amd)
random_disc_mean, random_disc_std = np.mean(random_disc), np.std(random_disc)

print(f"\nRandom Sampling ({N_REPEATS} realisations):")
print(f"  AMD:          {random_amd_mean:.4f} ± {random_amd_std:.4f}")
print(f"  Discrepancy:  {random_disc_mean:.4f} ± {random_disc_std:.4f}")

### Sobol Sampling

In [ ]:
sobol_samples = sobol_sampling(N_SAMPLES, N_DIMS)  # no seed — Sobol is deterministic
sobol_amd     = average_minimum_distance(sobol_samples)
sobol_disc    = l2_star_discrepancy(sobol_samples)

# WARNING: Sobol requires N_SAMPLES to be a power of 2 for accurate evaluation
print(f"\nSobol Sequence:")
print(f"  AMD:          {sobol_amd:.4f}")
print(f"  Discrepancy:  {sobol_disc:.4f}")

### Latin Hypercube Sampling (LHS)

In [ ]:
lhs_amd  = []  # Average Minimum Distance across realisations
lhs_disc = []  # L2* Discrepancy across realisations

# LHS is stochastic, so multiple realisations are needed for reliable statistics
for i in range(N_REPEATS):
    samples = lhs_sampling(N_SAMPLES, N_DIMS, seed=SEED_BASE + i)  # different seed per realisation
    lhs_amd.append(average_minimum_distance(samples))
    lhs_disc.append(l2_star_discrepancy(samples))

lhs_amd_mean, lhs_amd_std   = np.mean(lhs_amd),  np.std(lhs_amd)
lhs_disc_mean, lhs_disc_std = np.mean(lhs_disc), np.std(lhs_disc)

print(f"\nLHS ({N_REPEATS} realisations):")
print(f"  AMD:          {lhs_amd_mean:.4f} ± {lhs_amd_std:.4f}")
print(f"  Discrepancy:  {lhs_disc_mean:.4f} ± {lhs_disc_std:.4f}")

## Visualise results

In [ ]:
# First 2 dimensions only — 4D space can't be visualised directly
random_pts = random_sampling(N_SAMPLES, N_DIMS, seed=SEED_BASE)
lhs_pts    = lhs_sampling(N_SAMPLES, N_DIMS, seed=SEED_BASE)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(random_pts[:, 0], random_pts[:, 1], s=10, alpha=0.7)
axes[0].set_title("Random Sampling")
axes[1].scatter(sobol_samples[:, 0], sobol_samples[:, 1], s=10, alpha=0.7)
axes[1].set_title("Sobol Sequence")
axes[2].scatter(lhs_pts[:, 0], lhs_pts[:, 1], s=10, alpha=0.7)
axes[2].set_title("Latin Hypercube Sampling")
plt.tight_layout()
plt.show()